# 거리뷰 이미지를 이용한 불법 주정차 분석

## 1) 거리뷰 이미지 수집

### 1. 구글 street view API 사용
- 거리뷰 이미지를 url로 바로 받아올 수 있음
- 단점: 상세한 거리뷰 이미지가 없음

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import time

load_dotenv()  
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

def check_streetview_available(lat, lng):
    """
    왜 메타데이터 API를 먼저 쓰냐:
    이미지 API는 없는 곳에 요청해도 과금될 수 있지만
    메타데이터 API는 무료이고 이미지 존재 여부를 먼저 확인할 수 있음
    """
    url = (
        f"https://maps.googleapis.com/maps/api/streetview/metadata"
        f"?location={lat},{lng}"
        f"&key={GOOGLE_API_KEY}"
    )
    res = requests.get(url)
    data = res.json()

    # status가 OK면 이미지 있음, ZERO_RESULTS면 없음
    return data.get("status") == "OK"

def fetch_streetview(lat, lng, zone_id, save_dir="streetview_images"):
    os.makedirs(save_dir, exist_ok=True)
    headings = [0, 90, 180, 270]
    saved_paths = []

    for heading in headings:
        url = (
            f"https://maps.googleapis.com/maps/api/streetview"
            f"?size=640x480"
            f"&location={lat},{lng}"
            f"&heading={heading}"
            f"&fov=90"
            f"&key={GOOGLE_API_KEY}"
        )
        res = requests.get(url)
        if res.status_code == 200:
            path = f"{save_dir}/{zone_id}_h{heading}.jpg"
            with open(path, 'wb') as f:
                f.write(res.content)
            saved_paths.append(path)
        time.sleep(0.2)

    return saved_paths

# 실행
safezone_df = pd.read_csv('./어린이보호구역_위치정보.csv')

available = []
unavailable = []

for idx, row in safezone_df.iterrows():
    lat, lng = row['위도'], row['경도']

    if check_streetview_available(lat, lng):
        paths = fetch_streetview(lat, lng, zone_id=idx)
        available.append(row['대상시설명'])
        print(f"✅ [{idx}] {row['대상시설명']} → {len(paths)}장 수집")
    else:
        unavailable.append(row['대상시설명'])
        print(f"❌ [{idx}] {row['대상시설명']} → 이미지 없음 스킵")

print(f"\n수집 성공: {len(available)}개 / 스킵: {len(unavailable)}개")
print(f"\n이미지 없는 구역: {unavailable}")

❌ [0] 배성유치원 → 이미지 없음 스킵
❌ [1] 이솔유치원 → 이미지 없음 스킵
❌ [2] 성모 유치원 → 이미지 없음 스킵
❌ [3] 판교샘유치원 → 이미지 없음 스킵
❌ [4] 건영장안유치원 → 이미지 없음 스킵
❌ [5] 뽀뽀뽀유치원 → 이미지 없음 스킵
❌ [6] 해나유치원 → 이미지 없음 스킵
❌ [7] 샛별유치원 → 이미지 없음 스킵
❌ [8] 성마르코유치원 → 이미지 없음 스킵
❌ [9] 숲리라유치원 → 이미지 없음 스킵
❌ [10] 즐거운유치원 → 이미지 없음 스킵
❌ [11] 혜성유치원 → 이미지 없음 스킵
❌ [12] 산성어린이집 → 이미지 없음 스킵
❌ [13] 성남어린이집 → 이미지 없음 스킵
❌ [14] 성현어린이집 → 이미지 없음 스킵
❌ [15] 리플플러스어린이집 → 이미지 없음 스킵
❌ [16] 당촌초등학교 → 이미지 없음 스킵
❌ [17] 수내초등학교 → 이미지 없음 스킵
❌ [18] 산성3어린이집 → 이미지 없음 스킵
❌ [19] 은서유치원 → 이미지 없음 스킵
❌ [20] 예원유치원 → 이미지 없음 스킵
❌ [21] 다솜유치원 → 이미지 없음 스킵
❌ [22] 아름다운유치원 → 이미지 없음 스킵
❌ [23] 세화유치원 → 이미지 없음 스킵
❌ [24] 분당중앙유치원 → 이미지 없음 스킵
❌ [25] 보듬이나눔이어린이집 → 이미지 없음 스킵
❌ [26] 운중하나어린이집 → 이미지 없음 스킵
❌ [27] 푸르니이매어린이집 → 이미지 없음 스킵
❌ [28] 성남혜은학교 → 이미지 없음 스킵
❌ [29] 성은특수학교 → 이미지 없음 스킵
❌ [30] 케이디엘피어학원 → 이미지 없음 스킵
❌ [31] 판교초등학교 → 이미지 없음 스킵
❌ [32] 성남동초등학교 → 이미지 없음 스킵
❌ [33] 태평초등학교 → 이미지 없음 스킵
❌ [34] 성수초등학교 → 이미지 없음 스킵
❌ [35] 금빛초등학교 → 이미지 없음 스킵
❌ [36] 왕남초등학교 → 이미지 없음 스킵
❌ [37] 검단초등학교 → 이미지 없음 스킵
❌ [38] 불곡초등

### 2. 카카오맵 API + Selenium을 통해 직접 수집

In [1]:
%pip install webdriver-manager


   ---------------------------------------- 2/2 [webdriver-manager]

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import os
import pandas as pd

SAVE_DIR = "./roadview_images"
os.makedirs(SAVE_DIR, exist_ok=True)

def get_kakao_roadview(lat, lng, facility_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--window-size=1280,720")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        url = f"https://map.kakao.com/link/roadview/{lat},{lng}"
        driver.get(url)
        time.sleep(4)

        if "준비 중" in driver.page_source:
            print(f"  ❌ [{facility_name}] 로드뷰 없음")
            return None

        # 파일명에 시설명 사용
        # 왜 replace('/', '_')냐: 시설명에 '/'가 있으면 경로로 인식돼서 오류 발생하기 때문
        safe_name = facility_name.replace('/', '_').replace(' ', '_')
        path = f"{SAVE_DIR}/{safe_name}_북쪽.png"
        driver.save_screenshot(path)
        print(f"  ✅ [{facility_name}] 저장 완료 → {path}")
        return path

    except Exception as e:
        print(f"  ❌ [{facility_name}] 오류: {e}")
        return None

    finally:
        driver.quit()


safezone_df = pd.read_csv(r'C:\Users\EL36\Desktop\1차프로젝트_CCTV\스쿨존목록_경기도.csv')

for idx, row in safezone_df.iterrows():
    print(f"[{idx}] {row['대상시설명']} 수집 중...")
    get_kakao_roadview(row['위도'], row['경도'], facility_name=row['대상시설명'])

[0] 광주광명초등학교 수집 중...
  ✅ [광주광명초등학교] 저장 완료 → ./roadview_images/광주광명초등학교_북쪽.png
[1] 양벌초등학교 수집 중...
  ✅ [양벌초등학교] 저장 완료 → ./roadview_images/양벌초등학교_북쪽.png
[2] 오포초등학교 수집 중...
  ✅ [오포초등학교] 저장 완료 → ./roadview_images/오포초등학교_북쪽.png
[3] 광주매곡초등학교 수집 중...
  ✅ [광주매곡초등학교] 저장 완료 → ./roadview_images/광주매곡초등학교_북쪽.png
[4] 광주도평초등학교 수집 중...
  ✅ [광주도평초등학교] 저장 완료 → ./roadview_images/광주도평초등학교_북쪽.png
[5] 도곡초등학교 수집 중...
  ✅ [도곡초등학교] 저장 완료 → ./roadview_images/도곡초등학교_북쪽.png
[6] 선동초등학교 수집 중...
  ✅ [선동초등학교] 저장 완료 → ./roadview_images/선동초등학교_북쪽.png
[7] 곤지암초등학교 수집 중...
  ✅ [곤지암초등학교] 저장 완료 → ./roadview_images/곤지암초등학교_북쪽.png
[8] 만선초등학교 수집 중...
  ✅ [만선초등학교] 저장 완료 → ./roadview_images/만선초등학교_북쪽.png
[9] 도궁초등학교 수집 중...
  ✅ [도궁초등학교] 저장 완료 → ./roadview_images/도궁초등학교_북쪽.png
[10] 도수초등학교 수집 중...
  ✅ [도수초등학교] 저장 완료 → ./roadview_images/도수초등학교_북쪽.png
[11] 광지원초등학교 수집 중...
  ✅ [광지원초등학교] 저장 완료 → ./roadview_images/광지원초등학교_북쪽.png
[12] 남한산초등학교 수집 중...
  ✅ [남한산초등학교] 저장 완료 → ./roadview_images/남한산초등학교_북쪽.png
[13] 경안초등학교 수집 중...
  ✅ [경안초등학교] 저

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import os
from collections import defaultdict

SAVE_DIR = "./roadview_images2"
os.makedirs(SAVE_DIR, exist_ok=True)

name_counter = defaultdict(int)


# ---------------------------
# 1. Driver 생성 (1회만)
# ---------------------------
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,720")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-infobars")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(20)
    return driver


# ---------------------------
# 2. 로드뷰 캡처 함수
# ---------------------------
def capture_roadview(driver, lat, lng, facility_name):

    url = f"https://map.kakao.com/link/roadview/{lat},{lng}"
    driver.get(url)

    try:
        # canvas 렌더 대기
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "canvas"))
        )

        time.sleep(1.2)  # 렌더 안정화

        # 🔹 미니맵 토글 (UI 방식)
        try:
            toggle = driver.find_element(By.ID, "view.deco.toggle")
            toggle.click()
            time.sleep(0.5)
        except:
            # 실패하면 강제 숨김
            driver.execute_script("""
                var el = document.getElementById('view.map');
                if(el){ el.style.display='none'; }
            """)

        # 🔹 파일명 정리
        safe_name = (
            str(facility_name)
            .replace('/', '_')
            .replace(' ', '_')
        )

        name_counter[safe_name] += 1
        number = name_counter[safe_name]

        filename = f"{safe_name}_{number:02d}.png"
        path = os.path.join(SAVE_DIR, filename)

        driver.save_screenshot(path)

        print(f"  ✅ [{facility_name}] 저장 완료 → {path}")
        return path

    except Exception as e:
        print(f"  ❌ [{facility_name}] 오류: {e}")
        return None


# ---------------------------
# 3. 실행부
# ---------------------------
def main():

    df = pd.read_csv(
        r'C:\Users\EL36\Desktop\1차프로젝트_CCTV\성남_어린이사고다발지_좌표.csv'
    )

    driver = create_driver()

    for idx, row in df.iterrows():
        print(f"[{idx}] {row['대상시설명']} 수집 중...")

        capture_roadview(
            driver,
            row['위도'],
            row['경도'],
            facility_name=row['대상시설명']
        )

    driver.quit()
    print("\n모든 작업 완료.")


if __name__ == "__main__":
    main()

[0] 경기도 성남시 수정구 태평동(금빛초교 부근) 수집 중...
  ✅ [경기도 성남시 수정구 태평동(금빛초교 부근)] 저장 완료 → ./roadview_images2\경기도_성남시_수정구_태평동(금빛초교_부근)_01.png
[1] 경기도 성남시 수정구 태평동(금빛초교 부근) 수집 중...
  ✅ [경기도 성남시 수정구 태평동(금빛초교 부근)] 저장 완료 → ./roadview_images2\경기도_성남시_수정구_태평동(금빛초교_부근)_02.png
[2] 경기도 성남시 수정구 태평동(금빛초교 부근) 수집 중...
  ❌ [경기도 성남시 수정구 태평동(금빛초교 부근)] 오류: Alert Text: 브라우저에서 소프트웨어 렌더링 모드를 사용 중입니다.
로드뷰는 하드웨어 가속 모드를 사용 시 더 나은 성능을 경험하실 수 있습니다.
Message: unexpected alert open: {Alert text : 브라우저에서 소프트웨어 렌더링 모드를 사용 중입니다.
로드뷰는 하드웨어 가속 모드를 사용 시 더 나은 성능을 경험하실 수 있습니다.}
  (Session info: chrome=145.0.7632.117)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x5b7dd3
	0x5b7e14
	0x3c1db0
	0x3cf609
	0x3d4df6
	0x3b0e8c
	0x3cec8e
	0x44b531
	0x42e7d6
	0x400049
	0x400e04
	0x816924
	0x811bf7
	0x82f5a0
	0x5d0f58
	0x5d891d
	0x5c0648
	0x5c0812
	0x5aa21a
	0x765c5d49
	0x7796d83b
	0x7796d7c1

[3] 경기도 성남시 수정구 태평동(금빛초교 부근) 수집 중...


UnexpectedAlertPresentException: Alert Text: 브라우저에서 소프트웨어 렌더링 모드를 사용 중입니다.
로드뷰는 하드웨어 가속 모드를 사용 시 더 나은 성능을 경험하실 수 있습니다.
Message: unexpected alert open: {Alert text : 브라우저에서 소프트웨어 렌더링 모드를 사용 중입니다.
로드뷰는 하드웨어 가속 모드를 사용 시 더 나은 성능을 경험하실 수 있습니다.}
  (Session info: chrome=145.0.7632.117)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x5b7dd3
	0x5b7e14
	0x3c1db0
	0x44bec4
	0x42e7d6
	0x400049
	0x400e04
	0x816924
	0x811bf7
	0x82f5a0
	0x5d0f58
	0x5d891d
	0x5c0648
	0x5c0812
	0x5aa21a
	0x765c5d49
	0x7796d83b
	0x7796d7c1


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import os
import pandas as pd
from collections import defaultdict

SAVE_DIR = "./danger_zones_images"
os.makedirs(SAVE_DIR, exist_ok=True)

# 🔹 시설명별 카운터 저장용
name_counter = defaultdict(int)

def get_kakao_roadview(lat, lng, facility_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--window-size=1280,720")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        url = f"https://map.kakao.com/link/roadview/{lat},{lng}"
        driver.get(url)
        time.sleep(4)

        if "준비 중" in driver.page_source:
            print(f"  ❌ [{facility_name}] 로드뷰 없음")
            return None

        # 파일명 안전 처리
        safe_name = (
            facility_name
            .replace('/', '_')
            .replace(' ', '_')
        )

        # 🔹 번호 증가
        name_counter[safe_name] += 1
        number = name_counter[safe_name]

        # 2자리 포맷
        filename = f"{safe_name}_{number:02d}.png"
        path = os.path.join(SAVE_DIR, filename)

        driver.save_screenshot(path)
        print(f"  ✅ [{facility_name}] 저장 완료 → {path}")
        return path

    except Exception as e:
        print(f"  ❌ [{facility_name}] 오류: {e}")
        return None

    finally:
        driver.quit()


danger_zone_df = pd.read_csv(
    r'C:\Users\EL36\Desktop\1차프로젝트_CCTV\CustomVision\전국_어린이사고다발지역.csv'
)

for idx, row in danger_zone_df.iterrows():
    print(f"[{idx}] {row['대상시설명']} 수집 중...")
    get_kakao_roadview(
        row['위도'],
        row['경도'],
        facility_name=row['대상시설명']
    )

[0] 전북특별자치도 정읍시 상동(한솔초등학교 부근) 수집 중...
  ✅ [전북특별자치도 정읍시 상동(한솔초등학교 부근)] 저장 완료 → ./danger_zones_images\전북특별자치도_정읍시_상동(한솔초등학교_부근)_01.png
[1] 제주특별자치도 제주시 일도이동(꼬마스쿨 어린이집 부근) 수집 중...
  ✅ [제주특별자치도 제주시 일도이동(꼬마스쿨 어린이집 부근)] 저장 완료 → ./danger_zones_images\제주특별자치도_제주시_일도이동(꼬마스쿨_어린이집_부근)_01.png
[2] 제주특별자치도 제주시 도남동(도남초등학교 부근) 수집 중...
  ✅ [제주특별자치도 제주시 도남동(도남초등학교 부근)] 저장 완료 → ./danger_zones_images\제주특별자치도_제주시_도남동(도남초등학교_부근)_01.png
[3] 제주특별자치도 제주시 연동(신제주초등학교입구교차로 부근) 수집 중...
  ✅ [제주특별자치도 제주시 연동(신제주초등학교입구교차로 부근)] 저장 완료 → ./danger_zones_images\제주특별자치도_제주시_연동(신제주초등학교입구교차로_부근)_01.png
[4] 제주특별자치도 제주시 노형동(세빛어린이집 부근) 수집 중...
  ✅ [제주특별자치도 제주시 노형동(세빛어린이집 부근)] 저장 완료 → ./danger_zones_images\제주특별자치도_제주시_노형동(세빛어린이집_부근)_01.png
[5] 경상남도 거제시 고현동(중곡초등학교 부근) 수집 중...
  ✅ [경상남도 거제시 고현동(중곡초등학교 부근)] 저장 완료 → ./danger_zones_images\경상남도_거제시_고현동(중곡초등학교_부근)_01.png
[6] 경상남도 김해시 외동(봉명초등학교 부근) 수집 중...
  ✅ [경상남도 김해시 외동(봉명초등학교 부근)] 저장 완료 → ./danger_zones_images\경상남도_김해시_외동(봉명초등학교_부근)_01.png
[7] 경상남도 창원시 성산구 반림동(반송중학교 부근) 수집 중...
  ✅ [경

In [13]:
import requests

url = "https://dapi.kakao.com/v2/maps/staticmap"

params = {
    "center": "37.5665,126.9780",
    "level": 3,
    "w": 600,
    "h": 400,
    "maptype": "satellite"
}

headers = {
    "Authorization": "KakaoAK {510729beab989ed393b08584468c6564}"
}

res = requests.get(url, params=params, headers=headers)

print(res.status_code)
print(res.headers.get("Content-Type"))

404
application/json; charset=utf-8
